In [1]:
import pandas as pd
file_path: str = "data/news_data.csv"

df = pd.read_csv(file_path, sep=';', header=None, names=['label', 'text'], quotechar="'")
df.head()

# Aufgabe 1: Leere Zeilen und Dublikate entfernen

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

print(df.shape)
df.head(10)


(1028, 2)


,label,text
0,Wirtschaft,"Die Gewerkschaft GPA-djp lanciert den ""All-in-..."
1,Sport,Franzosen verteidigen 2:1-Führung – Kritische ...
2,Web,Neues Video von Designern macht im Netz die Ru...
3,Sport,23-jähriger Brasilianer muss vier Spiele pausi...
4,International,Aufständische verwendeten Chemikalie bei Gefec...
5,Web,Bewährungs- und Geldstrafe für 26-Jährigen weg...
6,Sport,ÖFB-Teamspieler nur sechs Minuten nach seinem ...
7,Panorama,Ein 31-jähriger Polizist soll einer 42-Jährige...
8,International,18 Menschen verschleppt. Kabul – Nach einem Hu...
9,Web,Deutschland und Frankreich am stärksten von Lo...


In [2]:
# Aufgabe 2 Remove stopwords and convert the text to lowercase 

from nltk.corpus import stopwords       # für deutsche Stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


# deutsche stopwörter laden
stop_words = set(stopwords.words('german'))

def clean_text(s):
    s = s.lower()  # nur kleinbuchstaben
    tokens = s.split()  # einfach in wörter splitten
    tokens = [w for w in tokens if w not in stop_words]  # stopwords entfernen
    return ' '.join(tokens)  # wieder zusammenbauen

df['text'] = df['text'].apply(clean_text)
df.head(10)

,label,text
0,Wirtschaft,"gewerkschaft gpa-djp lanciert ""all-in-rechner""..."
1,Sport,franzosen verteidigen 2:1-führung – kritische ...
2,Web,neues video designern macht netz runde – schla...
3,Sport,23-jähriger brasilianer vier spiele pausieren ...
4,International,aufständische verwendeten chemikalie gefechten...
5,Web,bewährungs- geldstrafe 26-jährigen wegen auslä...
6,Sport,öfb-teamspieler sechs minuten tor beim 1:1 sun...
7,Panorama,31-jähriger polizist 42-jährigen knöchel gebro...
8,International,18 menschen verschleppt. kabul – hubschrauber-...
9,Web,deutschland frankreich stärksten locky betroff...


In [3]:
# Aufgabe 3 Encode the labels

from sklearn.preprocessing import LabelEncoder

# LabelEncoder initialisieren
label_encoder = LabelEncoder()

# Labels encodieren (in deiner Fall-Spalte 'category' oder was immer du verwendest)
df['encoded_labels'] = label_encoder.fit_transform(df['label'])

# Zeige die ersten paar Zeilen an
df.head(1000)


,label,text,encoded_labels
0,Wirtschaft,"gewerkschaft gpa-djp lanciert ""all-in-rechner""...",7
1,Sport,franzosen verteidigen 2:1-führung – kritische ...,5
2,Web,neues video designern macht netz runde – schla...,6
3,Sport,23-jähriger brasilianer vier spiele pausieren ...,5
4,International,aufständische verwendeten chemikalie gefechten...,2
...,...,...,...
995,Inland,70 prozent schätzen diesjährige matura schwer ...,1
996,Web,gut verarbeitetes gerät aktueller atom-plattfo...,6
997,Wissenschaft,frage perspektive: hubble-daten errechneten as...,8
998,Panorama,umzug asylwerber winterfestes quartier abgesch...,4


In [4]:
# Aufgabe 4 Create a train/test split (80/20%) with stratified labels

from sklearn.model_selection import train_test_split

# Features und Ziel definiert aus deinem DataFrame
X = df['text']
y = df['encoded_labels']

# Stratified Train/Test-Split: 80% Training, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% der Daten ins Test-Set
    random_state=42,    # sorgt für Reproduzierbarkeit
    stratify=y          # wahrt die Klassenverteilung
)

# Optional: Größenkontrolle
print(f"Trainingsdaten: {X_train.shape[0]} Zeilen")
print(f"Testdaten:      {X_test.shape[0]} Zeilen")

Trainingsdaten: 822 Zeilen
Testdaten:      206 Zeilen


In [5]:
# Aufgabe 5 train a logistic regression classifier with tf/idf feature vectors

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. TF-IDF Vektorisierung auf Trainingsdaten anpassen
tfidf = TfidfVectorizer(
    max_features=10000,      # optional: beschränkt die Dimension
    ngram_range=(1,2),       # Uni- und Bi-Gramme
    stop_words=None          # wir haben bereits stopwords entfernt
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# 2. Logistic Regression initialisieren und trainieren
lr = LogisticRegression(
    solver='lbfgs',          # guter Default-Solver
    max_iter=1000,           # genug Iterationen fürs Konvergieren
    random_state=42
)
lr.fit(X_train_tfidf, y_train)

# 3. Vorhersage auf dem Testset
y_pred = lr.predict(X_test_tfidf)

# 4. Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=3))


Accuracy: 0.6941747572815534

Classification Report:
               precision    recall  f1-score   support

           0      1.000     0.077     0.143        13
           1      0.812     0.619     0.703        21
           2      0.758     0.833     0.794        30
           3      1.000     0.091     0.167        11
           4      0.565     0.765     0.650        34
           5      1.000     0.917     0.957        24
           6      0.636     0.824     0.718        34
           7      0.600     0.857     0.706        28
           8      1.000     0.273     0.429        11

    accuracy                          0.694       206
   macro avg      0.819     0.584     0.585       206
weighted avg      0.759     0.694     0.661       206



In [6]:
# Aufgabe 5 

# 1. TF-IDF Vektorisierung
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words=None)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# 2. Logistic Regression trainieren
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)

# 3. Test-Evaluation
y_pred = lr.predict(X_test_tfidf)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nTest Classification Report:\n", classification_report(y_test, y_pred, digits=3))

# 4. Training-Evaluation
y_train_pred = lr.predict(X_train_tfidf)
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("\nTraining Classification Report:\n", classification_report(y_train, y_train_pred, digits=3))

Test Accuracy: 0.6941747572815534

Test Classification Report:
               precision    recall  f1-score   support

           0      1.000     0.077     0.143        13
           1      0.812     0.619     0.703        21
           2      0.758     0.833     0.794        30
           3      1.000     0.091     0.167        11
           4      0.565     0.765     0.650        34
           5      1.000     0.917     0.957        24
           6      0.636     0.824     0.718        34
           7      0.600     0.857     0.706        28
           8      1.000     0.273     0.429        11

    accuracy                          0.694       206
   macro avg      0.819     0.584     0.585       206
weighted avg      0.759     0.694     0.661       206

Training Accuracy: 0.9647201946472019

Training Classification Report:
               precision    recall  f1-score   support

           0      1.000     0.796     0.887        54
           1      0.988     0.975     0.981       

In [7]:
# Aufgabe 6 train 2 other scikit learn models (e.g. RandomForestClassifier and LinearSVC)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=100,       # Anzahl der Bäume
    max_depth=None,         # keine maximale Tiefe
    random_state=42,
    class_weight='balanced' # um mit möglichen Klassenungleichgewichten umzugehen
)
rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)

print("--- Random Forest ---")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Macro-F1:", f1_score(y_test, y_pred_rf, average='macro'))
print("Micro-F1:", f1_score(y_test, y_pred_rf, average='micro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, digits=3))

# --- Linear SVC ---
svc = LinearSVC(
    C=1.0,                  # Regularisierungsparameter
    max_iter=10000,
    class_weight='balanced',# verbessert ggf. Performance bei ungleichen Klassen
    random_state=42
)
svc.fit(X_train_tfidf, y_train)
y_pred_svc = svc.predict(X_test_tfidf)

print("\n--- Linear SVC ---")
print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print("Macro-F1:", f1_score(y_test, y_pred_svc, average='macro'))
print("Micro-F1:", f1_score(y_test, y_pred_svc, average='micro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svc, digits=3))

--- Random Forest ---
Accuracy: 0.6699029126213593
Macro-F1: 0.6532613029471952
Micro-F1: 0.6699029126213593

Classification Report:
               precision    recall  f1-score   support

           0      0.444     0.308     0.364        13
           1      0.524     0.524     0.524        21
           2      0.688     0.733     0.710        30
           3      0.667     0.545     0.600        11
           4      0.567     0.500     0.531        34
           5      0.957     0.917     0.936        24
           6      0.667     0.824     0.737        34
           7      0.645     0.714     0.678        28
           8      0.889     0.727     0.800        11

    accuracy                          0.670       206
   macro avg      0.672     0.644     0.653       206
weighted avg      0.667     0.670     0.665       206


--- Linear SVC ---
Accuracy: 0.8009708737864077
Macro-F1: 0.7904220726373395
Micro-F1: 0.8009708737864077

Classification Report:
               precision    re

In [8]:
# Aufgabe 7 Compare results based on accuracy and macro/micro f1-score

from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# Modelle und ihre Test-Vorhersagen in Listen
models = [
    ("Logistic Regression", lr, y_pred),
    ("Random Forest",         rf, y_pred_rf),
    ("Linear SVC",            svc, y_pred_svc)
]

# Ergebnisse sammeln
results = []
for name, model, y_pred_model in models:
    acc     = accuracy_score(y_test, y_pred_model)
    macro_f = f1_score(y_test, y_pred_model, average='macro')
    micro_f = f1_score(y_test, y_pred_model, average='micro')
    results.append({
        "Modell": name,
        "Accuracy": acc,
        "Macro-F1": macro_f,
        "Micro-F1": micro_f
    })

# Als DataFrame anzeigen
results_df = pd.DataFrame(results)
print(results_df)


                Modell  Accuracy  Macro-F1  Micro-F1
0  Logistic Regression  0.694175  0.584978  0.694175
1        Random Forest  0.669903  0.653261  0.669903
2           Linear SVC  0.800971  0.790422  0.800971
